# Pipeline de Unión de *datasets* de registros socioeconímicos

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener datos históricos anuales de la *Eurostat* (el portal de *datasets* estadísticos de la Unión Europea). El objetivo es consolidar una base de datos anuales sobre diferentes aspectos socioeconómicos de cada ciudad, como pueden ser:

* La pablación anual.
* El PIB (Producto Interior Bruto).
* Parque de vehículos.

En este *pipeline*, tratamos los siguientes problemas:

### El problema de los Nombres (Estandarización)

Eurostat no es consistente con los nombres de cada ciudad. Por ejemplo, para Madrid, a veces se usa "Madrid", otras "ES30" y otras "Comunidad de Madrid". Por tanto, creamos una función que "limpia" el texto al que hace referencia cada nombre.

### Integración mediante Merges

En lugar de copiar y pegar, usamos la lógica de bases de datos (*merge*). Usamos Ciudad y Años. Para que un dato se pegue a otro, ambos campos deben coincidir exactamente. Por eso la limpieza de nombres del punto 1 es el paso más importante de todo el código.

In [1]:
# Importación de librerías

import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

# =============================================================================
# CONFIGURACIÓN DE RUTAS DE ACCESO
# =============================================================================

# Definimos la ruta base de datasets.
# La carpeta "datasets" está un nivel por encima de la carpeta actual del proyecto.
BASE_PATH = Path("..", "..") / "datasets"

# Carpeta donde se encuentran los archivos socioeconómicos
SOCIO_DIR = BASE_PATH / "archivos_socioeconómicos"

# Creamos la carpeta si no existe
SOCIO_DIR.mkdir(parents=True, exist_ok=True)

# Definimos las rutas de entrada
ruta_pob = SOCIO_DIR / "población.csv"
ruta_PIB = SOCIO_DIR / "PIB.csv"
ruta_vehiculos = SOCIO_DIR / "vehículos.csv"

# Definimos la ruta de salida
ruta_salida = SOCIO_DIR / "dataset_socioeconómico.csv"
print(f"📄 Archivo de salida: {ruta_salida}")

📄 Archivo de salida: ..\..\datasets\archivos_socioeconómicos\dataset_socioeconómico.csv


In [3]:
# =============================================================================
# CARGA DE DATOS (LECTURA)
# =============================================================================
# Leemos los CSV especificando:
# - sep=';' -> El delimitador que usan tus archivos.
# - encoding='latin1' -> Necesario para leer caracteres europeos (ñ, acentos, letras nórdicas).
# - encoding='utf-8-sig' -> Específico para el de Andorra para eliminar el carácter BOM invisible al inicio.

df_pob = pd.read_csv(ruta_pob, sep=';', encoding='latin1')
df_pib = pd.read_csv(ruta_PIB, sep=';', encoding='latin1')
df_veh = pd.read_csv(ruta_vehiculos, sep=';', encoding='latin1')

In [4]:
# =============================================================================
# FUNCIÓN DE TRANSFORMACIÓN (LÓGICA DE LIMPIEZA) - VERSIÓN FINAL
# =============================================================================
def transformar_nombres(df, col):
    """
    Función para limpiar columnas de ubicación.
    1. Copia el DF para evitar advertencias de SettingWithCopy.
    2. Elimina la terminación ' (greater city)'.
    3. Elimina nombres bilingües tras la barra (ej. '/València').
    4. Estandariza 'Tenerife' a 'Santa Cruz de Tenerife'.
    """
    df = df.copy()
    
    def limpiar(valor):
        if not isinstance(valor, str): return valor
        
        # Elimina espacios en blanco básicos
        valor = valor.strip() 
            
        # Eliminar la terminación específica de Eurostat (greater city)
        valor = valor.replace(" (greater city)", "").strip()
        
        # Eliminar el nombre bilingüe tras la barra (ej. "Valencia/València" -> "Valencia")
        # split("/") corta por la barra y [0] se queda con la primera parte
        if "/" in valor:
            valor = valor.split("/", 1)[0].strip()

        # Estandarización específica de Tenerife
        # Si el valor es exactamente 'Tenerife', lo convertimos al nombre oficial
        if valor == "Tenerife":
            valor = "Santa Cruz de Tenerife"

        if valor == "A CoruÃ±a":
            valor = "A Coruña"

        return valor

    df[col] = df[col].apply(limpiar)
    return df

In [5]:
# =============================================================================
# TRATAMIENTO DE LOS DATOS DE POBLACIÓN
# =============================================================================

# Extraemos columnas y renombramos
df_pob = df_pob[['cities', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_pob.columns = ['Ciudad', 'Años', 'Población']

# Limpieza de Nombres (usando tu función transformada)
df_pob = transformar_nombres(df_pob, col="Ciudad")

# Convertimos a entero de forma segura
# Usamos 'Int64' (con I mayúscula) que es un tipo de dato de Pandas 
# que sí permite enteros con algunos valores nulos si aparecieran después.
df_pob["Años"] = pd.to_numeric(df_pob["Años"], errors='coerce').astype('Int64')

In [6]:
# =============================================================================
# TRATAMIENTO DE LOS DATOS DE PIB
# =============================================================================

# Extraemos columnas y renombramos
df_pib = df_pib[['geo', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_pib.columns = ['Ciudad', 'Años', 'PIB']

# Limpieza de Nombres (usando tu función transformada)
df_pib = transformar_nombres(df_pib, col="Ciudad")

# 4. Convertimos a entero de forma segura
# Usamos 'Int64' (con I mayúscula) que es un tipo de dato de Pandas 
# que sí permite enteros con algunos valores nulos si aparecieran después.
df_pib["Años"] = pd.to_numeric(df_pib["Años"], errors='coerce').astype('Int64')

In [7]:
# =============================================================================
# TRATAMIENTO DE LOS DATOS DE VEHÍCULOS
# =============================================================================

# Extraemos columnas y renombramos
df_veh = df_veh[['cities', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_veh.columns = ['Ciudad', 'Años', 'Vehiculos_1000_hab']

# Limpieza de Nombres (usando tu función transformada)
df_veh = transformar_nombres(df_veh, col="Ciudad")

# Convertimos a entero de forma segura
# Usamos 'Int64' (con I mayúscula) que es un tipo de dato de Pandas 
# que sí permite enteros con algunos valores nulos si aparecieran después.
df_veh["Años"] = pd.to_numeric(df_veh["Años"], errors='coerce').astype('Int64')
df_veh["Vehiculos_1000_hab"] = pd.to_numeric(df_veh["Vehiculos_1000_hab"].astype(str).str.replace(',', '.'), errors='coerce').astype(float)

In [8]:
# =============================================================================
# MERGEO DE TABLAS
# =============================================================================
# Paso 1: Merge entre población y PIB
df_merged = pd.merge(df_pob, df_pib, on=['Ciudad', 'Años'], how='outer')

# Paso 2: Merge del resultado anterior con vehículos
df_final = pd.merge(df_merged, df_veh, on=['Ciudad', 'Años'], how='outer')

In [9]:
# Nos descargamos el dataset final
df_final.to_csv(ruta_salida, sep=';', decimal=',', index=False, encoding='utf-8-sig')